In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth  # Do this in local & cloud setups
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
    !pip install --no-deps --upgrade "torchao>=0.16.0"
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [2]:
!sudo apt-get install -qq libomp-dev
!pip install -qq faiss-gpu-cu12


debconf: unable to initialize frontend: Dialog
debconf: (No usable dialog-like program is installed, so the dialog based frontend cannot be used. at /usr/share/perl5/Debconf/FrontEnd/Dialog.pm line 78, <> line 4.)
debconf: falling back to frontend: Readline
debconf: unable to initialize frontend: Readline
debconf: (This frontend requires a controlling tty.)
debconf: falling back to frontend: Teletype
dpkg-preconfigure: unable to re-open stdin: 
Selecting previously unselected package libllvm14:amd64.
(Reading database ... 122354 files and directories currently installed.)
Preparing to unpack .../libllvm14_1%3a14.0.0-1ubuntu1.1_amd64.deb ...
Unpacking libllvm14:amd64 (1:14.0.0-1ubuntu1.1) ...
Selecting previously unselected package libomp5-14:amd64.
Preparing to unpack .../libomp5-14_1%3a14.0.0-1ubuntu1.1_amd64.deb ...
Unpacking libomp5-14:amd64 (1:14.0.0-1ubuntu1.1) ...
Selecting previously unselected package libomp-14-dev.
Preparing to unpack .../libomp-14-dev_1%3a14.0.0-1ubuntu1.1_am

In [3]:
!pip install -U bitsandbytes>=0.46.1

In [4]:
import numpy as np
import collections
import torch
import faiss
# import evaluate
from unsloth import FastLanguageModel
from transformers import pipeline
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel
from transformers import AutoModelForQuestionAnswering
from transformers import TrainingArguments
from transformers import Trainer
from tqdm.auto import tqdm

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [5]:
DATASET_NAME = 'squad_v2'
raw_datasets = load_dataset(DATASET_NAME, split='train')
raw_datasets

README.md: 0.00B [00:00, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 130319
})

In [6]:
raw_datasets = raw_datasets.filter(
    lambda x: len(x['answers']['text']) > 0
)
raw_datasets

Filter:   0%|          | 0/130319 [00:00<?, ? examples/s]

Dataset({
    features: ['id', 'title', 'context', 'question', 'answers'],
    num_rows: 86821
})

In [7]:
columns = raw_datasets.column_names
columns_to_keep = ['id', 'context', 'question', 'answers']
columns_to_remove = set(columns_to_keep).symmetric_difference(columns)
raw_datasets = raw_datasets.remove_columns(columns_to_remove)
raw_datasets

Dataset({
    features: ['id', 'context', 'question', 'answers'],
    num_rows: 86821
})

In [8]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [9]:
def cls_pooling(model_output):
    return model_output.last_hidden_state[:, 0]

def get_embeddings(text_list):
    encoded_input = tokenizer(
        text_list,
        padding=True,
        truncation=True,
        return_tensors='pt'
    )
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    model_output = model(**encoded_input)

    return cls_pooling(model_output)

In [10]:
embedding = get_embeddings(raw_datasets['question'][0])
embedding.shape

torch.Size([1, 768])

In [11]:
# Convert to numpy array (required for HF Datasets)
EMBEDDING_COLUMN = 'question_embedding'
embeddings_dataset = raw_datasets.map(
    lambda x: {EMBEDDING_COLUMN: get_embeddings(x['question']).detach().cpu().numpy()[0]}
)

Map:   0%|          | 0/86821 [00:00<?, ? examples/s]

In [12]:
embeddings_dataset.add_faiss_index(column=EMBEDDING_COLUMN)

  0%|          | 0/87 [00:00<?, ?it/s]

Dataset({
    features: ['id', 'context', 'question', 'answers', 'question_embedding'],
    num_rows: 86821
})

In [13]:
question = 'When did Beyonce start becoming popular?'

input_quest_embedding = get_embeddings([question]).cpu().detach().numpy()
input_quest_embedding.shape

# Output: (1, 768)

(1, 768)

In [14]:
TOP_K = 5
scores, samples = embeddings_dataset.get_nearest_examples(
    EMBEDDING_COLUMN, input_quest_embedding, k=TOP_K
)

# --- BẠN THÊM ĐOẠN NÀY VÀO DƯỚI ---
print("TÌM THẤY 5 KẾT QUẢ GẦN NHẤT:\n" + "="*50)

for i in range(TOP_K):
    print(f" Top {i + 1} (Score): {scores[i]:.2f})")
    print(f" Context: {samples['context'][i]}")
    print(f" Question: {samples['question'][i]}")
    print("-" * 50)


TÌM THẤY 5 KẾT QUẢ GẦN NHẤT:
 Top 1 (Score): 0.00)
 Context: Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in various singing and dancing competitions as a child, and rose to fame in the late 1990s as lead singer of R&B girl-group Destiny's Child. Managed by her father, Mathew Knowles, the group became one of the world's best-selling girl groups of all time. Their hiatus saw the release of Beyoncé's debut album, Dangerously in Love (2003), which established her as a solo artist worldwide, earned five Grammy Awards and featured the Billboard Hot 100 number-one singles "Crazy in Love" and "Baby Boy".
 Question: When did Beyonce start becoming popular?
--------------------------------------------------
 Top 2 (Score): 2.61)
 Context: Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, son

In [21]:
from unsloth import FastLanguageModel
from transformers import pipeline
import torch

MODEL_NAME = 'MinhQuy24/llama3.2_3B_SQuAD_QA'
max_seq_length = 2048
dtype = None # Tự động phát hiện (Float16 cho Tesla T4, Bfloat16 cho Ampere)
load_in_4bit = True

# 1. Load model bằng Unsloth (đúng chuẩn với lúc bạn save)
model_QA, tokenizer_QA = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)
text_streamer = TextStreamer(tokenizer_QA, skip_prompt = True)
FastLanguageModel.for_inference(model)


==((====))==  Unsloth 2026.5.2: Fast Llama patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.34. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


DistilBertModel(
  (embeddings): Embeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (transformer): Transformer(
    (layer): ModuleList(
      (0-5): 6 x TransformerBlock(
        (attention): DistilBertSdpaAttention(
          (dropout): Dropout(p=0.1, inplace=False)
          (q_lin): Linear(in_features=768, out_features=768, bias=True)
          (k_lin): Linear(in_features=768, out_features=768, bias=True)
          (v_lin): Linear(in_features=768, out_features=768, bias=True)
          (out_lin): Linear(in_features=768, out_features=768, bias=True)
        )
        (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (ffn): FFN(
          (dropout): Dropout(p=0.1, inplace=False)
          (lin1): Linear(in_features=768, out_features=3072, bias=True)
          (lin2): L

In [22]:
question = 'When did Beyonce start becoming popular?'

input_quest_embedding = get_embeddings([question]).cpu().detach().numpy()
input_quest_embedding.shape

# Output mong đợi: (1, 768)

(1, 768)

In [23]:
def format_test_squad(context,question):
    return {"text": f"""<|begin_of_text|>
<|start_header_id|>system<|end_header_id|>
You are a helpful AI assistant providing accurate answers based on the given context.
<|eot_id|>
<|start_header_id|>user<|end_header_id|>
Context: {context}
Question: {question}
<|eot_id|>
<|start_header_id|>assistant<|end_header_id|>
Answer: """}

In [24]:
from transformers import TextStreamer

In [28]:
TOP_K = 3
for idx, input_question in enumerate(embeddings_dataset['question'][200:210]):
    input_quest_embedding = get_embeddings([input_question]).cpu().detach().numpy()
    scores, samples = embeddings_dataset.get_nearest_examples(
        EMBEDDING_COLUMN, input_quest_embedding, k=TOP_K
    )

    print(f'Question {idx + 1}: {input_question}')
    for jdx, score in enumerate(scores):
        print(f'Top {jdx + 1}\tScore: {score}')
        question = samples['question'][jdx]
        context = samples['context'][jdx]

        formatted_prompt = format_test_squad(context, question)["text"]

        inputs = tokenizer_QA(
            formatted_prompt,
            return_tensors = "pt",
        ).to("cuda")

        # Tạo output tokens
        outputs = model_QA.generate(
            input_ids = inputs["input_ids"],
            max_new_tokens = 64,
            use_cache = True,
            temperature = 1.5,
            min_p = 0.1
        )

        # Decode và xóa <|eot_id|>
        full_text = tokenizer_QA.decode(outputs[0], skip_special_tokens=False)
        # Lấy phần assistant response (sau prompt)
        answer = full_text.split("Answer: ")[-1].replace('<|eot_id|>', '').strip()

        print(f'Context: {context}')
        print(f'Answer: {answer}')
        print()
    print()

Question 1: How many awards was Beyonce nominated for at the 52nd Grammy Awards?
Top 1	Score: 0.0
Context: At the 52nd Annual Grammy Awards, Beyoncé received ten nominations, including Album of the Year for I Am... Sasha Fierce, Record of the Year for "Halo", and Song of the Year for "Single Ladies (Put a Ring on It)", among others. She tied with Lauryn Hill for most Grammy nominations in a single year by a female artist. In 2010, Beyoncé was featured on Lady Gaga's single "Telephone" and its music video. The song topped the US Pop Songs chart, becoming the sixth number-one for both Beyoncé and Gaga, tying them with Mariah Carey for most number-ones since the Nielsen Top 40 airplay chart launched in 1992. "Telephone" received a Grammy Award nomination for Best Pop Collaboration with Vocals.
Answer: 10 nominations

Top 2	Score: 2.384733200073242
Context: At the 57th Annual Grammy Awards in February 2015, Beyoncé was nominated for six awards, ultimately winning three: Best R&B Performanc

In [ ]:
save_path = "MinhQuy24/SQuAD_QA_Vector_Database"
HF_KEY=''

In [33]:
# 1. Đảm bảo index đã được khởi tạo
if not embeddings_dataset.is_index_initialized(EMBEDDING_COLUMN):
    print(f"Khởi tạo lại index cho cột {EMBEDDING_COLUMN}...")
    embeddings_dataset.add_faiss_index(column=EMBEDDING_COLUMN)

# 2. Lưu FAISS index ra file cục bộ
embeddings_dataset.save_faiss_index(EMBEDDING_COLUMN, "my_index.faiss")

# 3. Gỡ index để push dataset lên Hub
embeddings_dataset.drop_index(EMBEDDING_COLUMN)
embeddings_dataset.push_to_hub(save_path, token = HF_KEY)

# 4. Sử dụng HfApi để upload file index riêng biệt
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="my_index.faiss",
    path_in_repo="my_index.faiss",
    repo_id=save_path,
    repo_type="dataset",
    token = HF_KEY
)
print("Đã upload thành công dataset và index!")

Khởi tạo lại index cho cột question_embedding...


  0%|          | 0/87 [00:00<?, ?it/s]

Uploading the dataset shards:   0%|          | 0/1 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/4 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

                              :   0%|          |  557kB /  285MB            

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  my_index.faiss              :   1%|          | 1.96MB /  267MB            

Đã upload thành công dataset và index!


In [34]:
from datasets import load_dataset
from huggingface_hub import hf_hub_download
reloaded_dataset = load_dataset(save_path, split="train", token=HF_KEY)
index_file_path = hf_hub_download(
    repo_id=save_path,
    filename="my_index.faiss",
    repo_type="dataset",
    token=HF_KEY
)
reloaded_dataset.load_faiss_index('question_embedding', index_file_path)


README.md:   0%|          | 0.00/508 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/285M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/86821 [00:00<?, ? examples/s]

my_index.faiss:   0%|          | 0.00/267M [00:00<?, ?B/s]

In [37]:

input_question = "Beyonce nổi tiếng khi nào?"
input_quest_embedding = get_embeddings([input_question]).cpu().detach().numpy()

# Sau đó mới tìm kiếm trên dataset đã tải về
scores, samples = reloaded_dataset.get_nearest_examples(
    'question_embedding', input_quest_embedding, k=5
)
for i in range(5):
    print(f" Top {i + 1} (Score): {scores[i]:.2f})")
    print(f" Context: {samples['context'][i]}")
    print(f" Question: {samples['question'][i]}")
    print("-" * 50)

 Top 1 (Score): 23.02)
 Context: The 50th anniversary of the novel's release was met with celebrations and reflections on its impact. Eric Zorn of the Chicago Tribune praises Lee's "rich use of language" but writes that the central lesson is that "courage isn't always flashy, isn't always enough, but is always in style". Jane Sullivan in the Sydney Morning Herald agrees, stating that the book "still rouses fresh and horrified indignation" as it examines morality, a topic that has recently become unfashionable. Chimamanda Ngozi Adichie writing in The Guardian states that Lee, rare among American novelists, writes with "a fiercely progressive ink, in which there is nothing inevitable about racism and its very foundation is open to question", comparing her to William Faulkner, who wrote about racism as an inevitability. Literary critic Rosemary Goring in Scotland's The Herald notes the connections between Lee and Jane Austen, stating the book's central theme, that "one’s moral convictions